In [1]:
# API-key setup — DO NOT hard-code your key in this cell.
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content, response.usage

answer, usage = ask_llm("What is the name of the president of Cameroon?")
print(answer)
print(usage)

The President of Cameroon is Paul Biya. He has been in office since November 6, 1982.
CompletionUsage(completion_tokens=24, prompt_tokens=51, total_tokens=75, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041845821, prompt_time=0.001693052, completion_time=0.05923256, total_time=0.060925612)


# Part 1.1 : Anatomy of a call

## 1. System vs user role: 
The system prompt is where you set up how the model should behave for the whole conversation, its role, tone, or any constraints you want it to follow. It's instructions for the model itself, not really something the model is "replying" to. For example, in this lab a good system prompt might be something like "You are a financial analyst assistant that only summarizes facts stated in the application, and never assumes anything the applicant didn't say." That sets the ground rules before any real question comes in.

The user role is the actual input or question you're asking the model to respond to, like the loan application text itself, or "Summarize this application in three sentences." It changes every time you call the model, while the system prompt usually stays the same across many calls.

So basically: system = the personality and rules, user = the specific request.

## 2. What is a token

A token is roughly a chunk of text, sometimes a whole word, sometimes part of a word, sometimes just punctuation. For example "microfinance" might get split into two tokens like "micro" and "finance" depending on how common the word is in the model's training data. Common short words like "the" or "is" are usually one token each.

Providers bill per token instead of per request because the actual cost to them scales with how much text the model has to process and generate, not with how many times you hit the API. A request asking for a one sentence answer costs way less compute than a request asking the model to read a 2,000 word loan application and generate a full page summary, even though both are technically "one request." 

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
question = "Suggest a name for a savings product for market traders in Accra."

low_temp_answers = []
for i in range(5):
    answer, _ = ask_llm(question, temperature=0.0)
    low_temp_answers.append(answer)

high_temp_answers = []
for i in range(5):
    answer, _ = ask_llm(question, temperature=1.2)
    high_temp_answers.append(answer)

# TODO: Print all 10 answers, grouped by temperature.
print("Temperature 0.0")
for i, ans in enumerate(low_temp_answers, 1):
    print(f"{i}. {ans}")

print("\nTemperature 1.2")
for i, ans in enumerate(high_temp_answers, 1):
    print(f"{i}. {ans}")

Temperature 0.0
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name emphasizes the idea of saving and accumulating wealth, which is a key goal for market traders.
3. **Accra Market Fund**: This name is straightforward and clearly communicates the product's purpose and target audience.
4. **Sika Kokoo**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Kokoo" means "gather" or "collect", so this name could encourage market traders to save and collect their earnings.
5. **Market Booster**: This name suggests that the savings product can help market traders boost their businesses and achieve their financial goals.
6. **Kae Dzi**: "Kae Dzi" is a Ghanaian phrase that means "save for the future". This name could appeal to market traders who are looking to plan for their future and secure th

# Part 1.2 Temperature: the randomness dial

## What did you observe at each temperature?
At temperature 0.0, I expected the five answers to be identical, but they weren't quite. Some names repeated in almost every run, like "Makola Save/Savings" and "Traders' Trust/Treasure", and runs 2 and 3 were word for word the same. But each run still had a few different names mixed in, so 0.0 was mostly consistent, not perfectly deterministic.
At temperature 1.2, the answers were clearly more varied. Each run had a mostly different set of names, some more unusual, like "SikaBox" and "Obaa Savings", that never showed up at 0.0. A few names like "Makola Savings" and "Kokroko Savings" still repeated, but overall there was much more spread than at 0.0.

## Which temperature is appropriate for the loan system?
Low temperature, close to 0.0, is the right choice. A loan officer needs the same application to produce the same summary, extracted data, and recommendation every time it's processed. Even the small variation I saw at 0.0 is something to minimize, not add to, so high temperature would only make the output less trustworthy for this use case.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [5]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this:"

v1_l002, _ = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L002']}")
v1_l006, _ = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L006']}")

print(" V1 - L002")
print(v1_l002)
print("\nV1 - L006")
print(v1_l006)

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer. You summarize loan "
    "applications factually and neutrally, in 3-4 sentences. Only use information "
    "explicitly stated in the letter. Do not invent, assume, or infer any details "
    "that are not written in the text, including amounts, dates, or motivations."
)

def summary_prompt_v2(letter_text):
    return f"Summarize this loan application:\n\n{letter_text}"

v2_l002, _ = ask_llm(summary_prompt_v2(LETTERS['L002']), system_prompt=SUMMARY_SYSTEM_V2, temperature=0)
v2_l006, _ = ask_llm(summary_prompt_v2(LETTERS['L006']), system_prompt=SUMMARY_SYSTEM_V2, temperature=0)

print("V2 - L002")
print(v2_l002)
print("\nV2 - L006")
print(v2_l006)

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("\n COMPARISON")
print("V1 L002:", v1_l002)
print("V2 L002:", v2_l002)
print()
print("V1 L006:", v1_l006)
print("V2 L006:", v2_l006)

 V1 - L002
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season, and is willing to repay the loan as soon as possible, despite not having any collateral to offer.

V1 - L006
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and promises to repay the loan within a year when his businesses are successful, relying on his trustworthiness.
V2 - L002
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects it to improve after the festive season. He does not currently have 

# Part 3.1 : Summarization prompts

## 1. What problems did V1 have that V2 fixed?

V1's L006 summary added "He has no experience", which is not actually stated in the letter. The letter only says he hasn't started the businesses yet, not that he lacks experience, that's a small inferred addition, not a stated fact. V1 also softened Kwame's vague repayment plan in L002 into "willing to repay the loan as soon as possible", when the letter actually just says "I can pay back whenever the money comes", a much weaker and less reassuring statement. V2 stuck closer to the source, for example writing "is requesting assistance with the loan" instead of adding a positive spin V1 implied.

V1 also had no consistent structure, both summaries read more like a narrative pitch than a factual brief, while V2 consistently opened with name, loan amount, and purpose in the same order across both letters, which is easier for a loan officer to scan quickly.

## 2. Why is "no invented details" essential, and what is this failure mode called?

If the model adds details not actually in the letter, like implying Kwame is more eager or reliable than he stated, or adding claims about Kofi's experience that aren't there, a loan officer reading only the summary could make a decision based on information the applicant never actually provided. In a financial context this could mean approving or misjudging a real loan based on a fabricated impression rather than the facts.

This failure mode is called hallucination in the LLM literature, when a model generates content that sounds plausible and fluent but is not grounded in the actual input or source material.